# PathSnap Data Exploration, Preprocessing, and Validation

This notebook combines the 21 numbered Python scripts in their original numeric order. It covers dataset inventory, GPS and road-network analysis, preprocessing, and validation.

**Run from top to bottom.** Later sections depend on reports and processed files created earlier. The original `.py` files remain unchanged.

All generated CSV files are saved under `csv_output/` inside the detected data folder, with report and processed files in subfolders.


In [27]:
from pathlib import Path

# Search the current folder, its parents, and common project data-folder layouts.
_search_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
_suffixes = [
    Path('.'),
    Path('data'),
    Path('ML_Hackathon') / 'data',
    Path('Student folder'),
    Path('Student folder') / 'Student folder',
    Path('ML_Hackathon'),
    Path('ML_Hackathon') / 'Student folder',
    Path('ML_Hackathon') / 'Student folder' / 'Student folder',
]
_candidates = [root / suffix for root in _search_roots for suffix in _suffixes]
_candidates.append(
    Path.home() / 'OneDrive' / 'Documents' / 'ML HACKATHON' / 'ML_Hackathon' / 'data'
)
DATA_DIR = next(
    (candidate.resolve() for candidate in _candidates
     if (candidate / 'train').is_dir() and (candidate / 'student_test').is_dir()),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError(
        'Could not find the data folder containing train/ and student_test/. '
        f'Jupyter is running from: {Path.cwd()}. Set DATA_DIR manually to the folder '
        'that directly contains train/ and student_test/.'
    )
CSV_OUTPUT_DIR = DATA_DIR / 'csv_output'
CSV_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Data folder: {DATA_DIR}')
print(f'CSV output folder: {CSV_OUTPUT_DIR}')


Data folder: C:\Users\vagee\OneDrive\Documents\ML HACKATHON\ML_Hackathon\data
CSV output folder: C:\Users\vagee\OneDrive\Documents\ML HACKATHON\ML_Hackathon\data\csv_output


## 01_dataset_inventory

Source: `01_dataset_inventory.py`

In [28]:

from pathlib import Path
import csv

BASE_DIR = DATA_DIR

PARTITIONS = ["train", "student_test"]

REQUIRED_EXTENSIONS = [
    ".track",
    ".nodes",
    ".arcs",
    ".route"
]

OUTPUT_DIR = BASE_DIR / "csv_output" / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

inventory = []


def count_rows(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        return sum(1 for line in file if line.strip())


for partition in PARTITIONS:

    partition_path = BASE_DIR / partition

    if not partition_path.exists():
        print(f"WARNING: {partition} folder not found")
        continue

    trip_folders = sorted(
        folder for folder in partition_path.iterdir()
        if folder.is_dir()
    )

    print(f"\nInspecting {partition}...")
    print(f"Number of trips: {len(trip_folders)}")

    for trip_folder in trip_folders:

        trip_id = trip_folder.name

        record = {
            "partition": partition,
            "trip_id": trip_id
        }

        for extension in REQUIRED_EXTENSIONS:

            file_path = trip_folder / f"{trip_id}{extension}"

            if file_path.exists():

                record[f"{extension}_exists"] = True
                record[f"{extension}_rows"] = count_rows(file_path)
                record[f"{extension}_size_kb"] = round(
                    file_path.stat().st_size / 1024, 2
                )

            else:

                record[f"{extension}_exists"] = False
                record[f"{extension}_rows"] = 0
                record[f"{extension}_size_kb"] = 0

        inventory.append(record)


if inventory:

    output_file = OUTPUT_DIR / "dataset_inventory.csv"

    fieldnames = list(inventory[0].keys())

    with open(output_file, "w", newline="", encoding="utf-8") as file:

        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(inventory)

    print("\n========== DATASET INVENTORY ==========")

    for partition in PARTITIONS:

        trips = [
            row for row in inventory
            if row["partition"] == partition
        ]

        print(f"\n{partition.upper()}")
        print(f"Total trips: {len(trips)}")

        for extension in REQUIRED_EXTENSIONS:

            total_rows = sum(
                row[f"{extension}_rows"] for row in trips
            )

            print(f"{extension} total rows: {total_rows}")

    print("\nInventory saved successfully!")
    print(f"Output: {output_file}")

else:
    print("No trip data was found.")



Inspecting train...
Number of trips: 80

Inspecting student_test...
Number of trips: 10

========== DATASET INVENTORY ==========

TRAIN
Total trips: 80
.track total rows: 208417
.nodes total rows: 1933485
.arcs total rows: 3876904
.route total rows: 31336

STUDENT_TEST
Total trips: 10
.track total rows: 19160
.nodes total rows: 90689
.arcs total rows: 185041
.route total rows: 3477

Inventory saved successfully!
Output: C:\Users\vagee\OneDrive\Documents\ML HACKATHON\ML_Hackathon\data\csv_output\reports\dataset_inventory.csv


## 02_load_raw_data

Source: `02_load_raw_data.py`

In [29]:

from pathlib import Path
import pandas as pd

BASE_DIR = DATA_DIR

TRIP_ID = "00000001"
PARTITION = "train"

TRIP_DIR = BASE_DIR / PARTITION / TRIP_ID

track_file = TRIP_DIR / f"{TRIP_ID}.track"
nodes_file = TRIP_DIR / f"{TRIP_ID}.nodes"
arcs_file = TRIP_DIR / f"{TRIP_ID}.arcs"
route_file = TRIP_DIR / f"{TRIP_ID}.route"

track = pd.read_csv(
    track_file,
    sep=r"\s+",
    header=None,
    names=["longitude", "latitude", "timestamp"]
)

nodes = pd.read_csv(
    nodes_file,
    sep=r"\s+",
    header=None,
    names=["longitude", "latitude"]
)

arcs = pd.read_csv(
    arcs_file,
    sep=r"\s+",
    header=None,
    names=["from_node_id", "to_node_id"]
)

route = pd.read_csv(
    route_file,
    sep=r"\s+",
    header=None,
    names=["arc_id"]
)

datasets = {
    "GPS Track": track,
    "Road Nodes": nodes,
    "Road Arcs": arcs,
    "Reference Route": route
}

for name, df in datasets.items():

    print(f"\n========== {name} ==========")

    print("\nFirst 5 rows:")
    print(df.head())

    print("\nShape:")
    print(df.shape)

    print("\nData types:")
    print(df.dtypes)

    print("\nMissing values:")
    print(df.isnull().sum())

print("\nAll four datasets loaded successfully!")



========== GPS Track ==========

First 5 rows:
   longitude   latitude  timestamp
0   8.595203  49.844053        1.0
1   8.595294  49.844441        2.0
2   8.595352  49.844883        3.0
3   8.595390  49.845379        4.0
4   8.595587  49.845750        5.0

Shape:
(1315, 3)

Data types:
longitude    float64
latitude     float64
timestamp    float64
dtype: object

Missing values:
longitude    0
latitude     0
timestamp    0
dtype: int64

========== Road Nodes ==========

First 5 rows:
   longitude   latitude
0   8.581679  49.857880
1   8.584637  49.858295
2   8.580173  49.858567
3   8.584392  49.859119
4   8.584935  49.857396

Shape:
(88809, 2)

Data types:
longitude    float64
latitude     float64
dtype: object

Missing values:
longitude    0
latitude     0
dtype: int64

========== Road Arcs ==========

First 5 rows:
   from_node_id  to_node_id
0             0         281
1             0           1
2             0           5
3             1           0
4             1       40750

S

## 03_data_quality_analysis

Source: `03_data_quality_analysis.py`

In [30]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE_DIR = DATA_DIR
TRIP_ID = "00000001"
PARTITION = "train"

TRIP_DIR = BASE_DIR / PARTITION / TRIP_ID

track = pd.read_csv(
    TRIP_DIR / f"{TRIP_ID}.track",
    sep=r"\s+",
    header=None,
    names=["longitude", "latitude", "timestamp"]
)

nodes = pd.read_csv(
    TRIP_DIR / f"{TRIP_ID}.nodes",
    sep=r"\s+",
    header=None,
    names=["longitude", "latitude"]
)

arcs = pd.read_csv(
    TRIP_DIR / f"{TRIP_ID}.arcs",
    sep=r"\s+",
    header=None,
    names=["from_node_id", "to_node_id"]
)

route = pd.read_csv(
    TRIP_DIR / f"{TRIP_ID}.route",
    sep=r"\s+",
    header=None,
    names=["arc_id"]
)

print("\n========== GPS DATA QUALITY ==========")

print("\nMissing values:")
print(track.isnull().sum())

print("\nDuplicate GPS observations:")
print(track.duplicated().sum())

print("\nInvalid longitude values:")
print((~track["longitude"].between(-180, 180)).sum())

print("\nInvalid latitude values:")
print((~track["latitude"].between(-90, 90)).sum())

print("\nNon-finite GPS values:")
print((~np.isfinite(track.select_dtypes(include="number"))).sum())

print("\nTimestamp analysis:")

time_diff = track["timestamp"].diff().dropna()

print("Minimum interval:", time_diff.min())
print("Maximum interval:", time_diff.max())
print("Average interval:", time_diff.mean())
print("Intervals not equal to 1 second:", (time_diff != 1).sum())
print("Non-increasing timestamps:", (time_diff <= 0).sum())

print("\n========== ROAD NODE VALIDATION ==========")

print("Total nodes:", len(nodes))

invalid_node_coordinates = (
    ~nodes["longitude"].between(-180, 180)
    | ~nodes["latitude"].between(-90, 90)
)

print("Invalid node coordinates:", invalid_node_coordinates.sum())

print("\n========== ROAD ARC VALIDATION ==========")

print("Total arcs:", len(arcs))

invalid_arcs = (
    (arcs["from_node_id"] < 0)
    | (arcs["from_node_id"] >= len(nodes))
    | (arcs["to_node_id"] < 0)
    | (arcs["to_node_id"] >= len(nodes))
)

print("Invalid arc connections:", invalid_arcs.sum())

print("\n========== REFERENCE ROUTE VALIDATION ==========")

print("Total route segments:", len(route))

invalid_route_ids = (
    (route["arc_id"] < 0)
    | (route["arc_id"] >= len(arcs))
)

print("Invalid route arc IDs:", invalid_route_ids.sum())

print("\n========== DATA QUALITY SUMMARY ==========")

print("GPS observations:", len(track))
print("Road nodes:", len(nodes))
print("Road arcs:", len(arcs))
print("Reference route segments:", len(route))

print("\nData quality analysis completed!")


========== GPS DATA QUALITY ==========

Missing values:
longitude    0
latitude     0
timestamp    0
dtype: int64

Duplicate GPS observations:
0

Invalid longitude values:
0

Invalid latitude values:
0

Non-finite GPS values:
longitude    0
latitude     0
timestamp    0
dtype: int64

Timestamp analysis:
Minimum interval: 1.0
Maximum interval: 16.0
Average interval: 1.0327245053272451
Intervals not equal to 1 second: 25
Non-increasing timestamps: 0

========== ROAD NODE VALIDATION ==========
Total nodes: 88809
Invalid node coordinates: 0

========== ROAD ARC VALIDATION ==========
Total arcs: 165790
Invalid arc connections: 0

========== REFERENCE ROUTE VALIDATION ==========
Total route segments: 525
Invalid route arc IDs: 0

========== DATA QUALITY SUMMARY ==========
GPS observations: 1315
Road nodes: 88809
Road arcs: 165790
Reference route segments: 525

Data quality analysis completed!


## 04_timestamp_analysis

Source: `04_timestamp_analysis.py`

In [31]:

from pathlib import Path
import pandas as pd
import numpy as np

BASE_DIR = DATA_DIR
TRIP_ID = "00000001"

track_file = (
    BASE_DIR / "train" / TRIP_ID / f"{TRIP_ID}.track"
)

track = pd.read_csv(
    track_file,
    sep=r"\s+",
    header=None,
    names=["longitude", "latitude", "timestamp"]
)


def haversine(lon1, lat1, lon2, lat2):

    earth_radius = 6371000

    lon1, lat1, lon2, lat2 = np.radians(
        [lon1, lat1, lon2, lat2]
    )

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

    return earth_radius * c


track["time_gap"] = track["timestamp"].diff()

track["distance_m"] = haversine(
    track["longitude"].shift(),
    track["latitude"].shift(),
    track["longitude"],
    track["latitude"]
)

track["speed_mps"] = (
    track["distance_m"] / track["time_gap"]
)

track["speed_kmh"] = track["speed_mps"] * 3.6

print("\n========== TIMESTAMP ANALYSIS ==========")

print("Total GPS observations:", len(track))
print("Total time intervals:", track["time_gap"].count())

print("Minimum time gap:", track["time_gap"].min())
print("Maximum time gap:", track["time_gap"].max())
print("Average time gap:", track["time_gap"].mean())

print("\n========== INTERVAL DISTRIBUTION ==========")

print(track["time_gap"].value_counts().sort_index())

print("\n========== GPS GAP DETAILS ==========")

gaps = track[
    track["time_gap"] > 1
].copy()

gaps["previous_timestamp"] = (
    track["timestamp"].shift()
)

gaps["previous_longitude"] = (
    track["longitude"].shift()
)

gaps["previous_latitude"] = (
    track["latitude"].shift()
)

gap_report = gaps[
    [
        "previous_timestamp",
        "timestamp",
        "time_gap",
        "previous_longitude",
        "previous_latitude",
        "longitude",
        "latitude",
        "distance_m",
        "speed_kmh"
    ]
]

print(gap_report.to_string(index=False))

print("\n========== SUMMARY ==========")

print("Normal 1-second intervals:",
      (track["time_gap"] == 1).sum())

print("Intervals greater than 1 second:",
      (track["time_gap"] > 1).sum())

print("Total GPS gaps:", len(gaps))

OUTPUT_DIR = BASE_DIR / "csv_output" / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

output_file = OUTPUT_DIR / "gps_gap_analysis.csv"

gap_report.to_csv(output_file, index=False)

print("\nGap analysis saved successfully!")
print("Output:", output_file)



========== TIMESTAMP ANALYSIS ==========
Total GPS observations: 1315
Total time intervals: 1314
Minimum time gap: 1.0
Maximum time gap: 16.0
Average time gap: 1.0327245053272451

========== INTERVAL DISTRIBUTION ==========
time_gap
1.0     1289
2.0       21
3.0        2
4.0        1
16.0       1
Name: count, dtype: int64

========== GPS GAP DETAILS ==========
 previous_timestamp  timestamp  time_gap  previous_longitude  previous_latitude  longitude  latitude  distance_m  speed_kmh
              948.0      950.0       2.0            8.647646          50.200242   8.647334 50.200070   29.307724  52.753903
              950.0      952.0       2.0            8.647334          50.200070   8.647132 50.199998   16.456483  29.621669
             1077.0     1079.0       2.0            8.604170          50.211513   8.604397 50.211350   24.278312  43.700962
             1186.0     1188.0       2.0            8.587983          50.199911   8.587757 50.199690   29.370846  52.867523
             118

## 05_road_network_analysis

Source: `05_road_network_analysis.py`

In [32]:

from pathlib import Path
import pandas as pd
import networkx as nx

BASE_DIR = DATA_DIR
TRIP_ID = "00000001"

TRIP_DIR = BASE_DIR / "train" / TRIP_ID

nodes = pd.read_csv(
    TRIP_DIR / f"{TRIP_ID}.nodes",
    sep=r"\s+",
    header=None,
    names=["longitude", "latitude"]
)

arcs = pd.read_csv(
    TRIP_DIR / f"{TRIP_ID}.arcs",
    sep=r"\s+",
    header=None,
    names=["from_node_id", "to_node_id"]
)

G = nx.DiGraph()

G.add_nodes_from(range(len(nodes)))

G.add_edges_from(
    arcs[["from_node_id", "to_node_id"]].itertuples(
        index=False,
        name=None
    )
)

components = list(nx.weakly_connected_components(G))

isolated_nodes = list(nx.isolates(G))

self_loops = list(nx.selfloop_edges(G))

duplicate_arcs = arcs.duplicated().sum()

component_sizes = sorted(
    [len(component) for component in components],
    reverse=True
)

print("\n========== ROAD NETWORK ANALYSIS ==========")

print("Total nodes:", len(nodes))
print("Total arcs:", len(arcs))

print("Unique connected nodes:", G.number_of_nodes())

print("Nodes with at least one connection:",
      sum(1 for node in G.nodes if G.degree(node) > 0))

print("Isolated nodes:", len(isolated_nodes))

print("Self-loops:", len(self_loops))

print("Duplicate directed arcs:", duplicate_arcs)

print("\n========== CONNECTED COMPONENTS ==========")

print("Number of weakly connected components:", len(components))

print("Largest component size:",
      component_sizes[0] if component_sizes else 0)

print("Smallest component size:",
      component_sizes[-1] if component_sizes else 0)

if components:

    largest_component = max(components, key=len)

    print("Largest component percentage:",
          round(len(largest_component) / len(nodes) * 100, 2), "%")

print("\n========== COMPONENT DISTRIBUTION ==========")

print("Top 10 component sizes:", component_sizes[:10])

OUTPUT_DIR = BASE_DIR / "csv_output" / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

component_report = pd.DataFrame({
    "component_id": range(len(component_sizes)),
    "number_of_nodes": component_sizes
})

component_report.to_csv(
    OUTPUT_DIR / "road_component_analysis.csv",
    index=False
)

print("\nRoad network analysis completed!")
print("Report saved successfully.")



========== ROAD NETWORK ANALYSIS ==========
Total nodes: 88809
Total arcs: 165790
Unique connected nodes: 88809
Nodes with at least one connection: 88809
Isolated nodes: 0
Self-loops: 0
Duplicate directed arcs: 292

========== CONNECTED COMPONENTS ==========
Number of weakly connected components: 175
Largest component size: 86854
Smallest component size: 2
Largest component percentage: 97.8 %

========== COMPONENT DISTRIBUTION ==========
Top 10 component sizes: [86854, 127, 80, 56, 49, 48, 46, 38, 37, 35]

Road network analysis completed!
Report saved successfully.


## 06_gps_trajectory_analysis

Source: `06_gps_trajectory_analysis.py`

In [33]:

from pathlib import Path

import numpy as np
import pandas as pd

BASE_DIR = DATA_DIR
OUTPUT_DIR = BASE_DIR / "csv_output" / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EARTH_RADIUS = 6371000

def haversine_distance(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(
        np.radians, [lon1, lat1, lon2, lat2]
    )

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )

    return 2 * EARTH_RADIUS * np.arcsin(np.sqrt(a))

results = []

for partition in ["train", "student_test"]:
    partition_dir = BASE_DIR / partition

    for trip_dir in sorted(partition_dir.iterdir()):
        if not trip_dir.is_dir():
            continue

        track_file = trip_dir / f"{trip_dir.name}.track"

        if not track_file.exists():
            continue

        track = pd.read_csv(
            track_file,
            sep=r"\s+",
            header=None,
            names=["longitude", "latitude", "timestamp"]
        )

        if len(track) < 2:
            continue

        distances = haversine_distance(
            track["longitude"].iloc[:-1].to_numpy(),
            track["latitude"].iloc[:-1].to_numpy(),
            track["longitude"].iloc[1:].to_numpy(),
            track["latitude"].iloc[1:].to_numpy()
        )

        time_differences = track["timestamp"].diff().iloc[1:].to_numpy()

        total_distance = distances.sum()

        valid_intervals = time_differences > 0
        total_time = time_differences[valid_intervals].sum()

        average_speed = (
            total_distance / total_time * 3.6
            if total_time > 0 else np.nan
        )

        results.append({
            "partition": partition,
            "trip_id": trip_dir.name,
            "gps_observations": len(track),
            "min_longitude": track["longitude"].min(),
            "max_longitude": track["longitude"].max(),
            "min_latitude": track["latitude"].min(),
            "max_latitude": track["latitude"].max(),
            "total_distance_km": total_distance / 1000,
            "average_time_interval_sec": time_differences.mean(),
            "average_speed_kmh": average_speed
        })

report = pd.DataFrame(results)

report.to_csv(
    OUTPUT_DIR / "gps_trajectory_analysis.csv",
    index=False
)

print("========== GPS TRAJECTORY ANALYSIS ==========")
print("Total trips analysed:", len(report))
print("\nTrips by partition:")
print(report["partition"].value_counts())

print("\nOverall statistics:")
print(report[[
    "gps_observations",
    "total_distance_km",
    "average_time_interval_sec",
    "average_speed_kmh"
]].describe().round(3))

print("\nReport saved successfully!")


========== GPS TRAJECTORY ANALYSIS ==========
Total trips analysed: 90

Trips by partition:
partition
train           80
student_test    10
Name: count, dtype: int64

Overall statistics:
       gps_observations  total_distance_km  average_time_interval_sec  \
count            90.000             90.000                     90.000   
mean           2528.633             36.393                      4.656   
std            3157.000             33.392                     30.262   
min             262.000              5.547                      1.000   
25%             899.250             12.765                      1.000   
50%            1496.500             24.066                      1.034   
75%            2816.250             48.763                      1.276   
max           19313.000            169.254                    287.907   

       average_speed_kmh  
count             90.000  
mean              49.147  
std               24.404  
min                0.201  
25%               33

## 07_gps_interval_quality

Source: `07_gps_interval_quality.py`

In [34]:

from pathlib import Path

import numpy as np
import pandas as pd

BASE_DIR = DATA_DIR
OUTPUT_DIR = BASE_DIR / "csv_output" / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EARTH_RADIUS = 6371000

def haversine_distance(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(
        np.radians, [lon1, lat1, lon2, lat2]
    )

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )

    return 2 * EARTH_RADIUS * np.arcsin(np.sqrt(a))

results = []

for partition in ["train", "student_test"]:
    partition_dir = BASE_DIR / partition

    for trip_dir in sorted(partition_dir.iterdir()):
        if not trip_dir.is_dir():
            continue

        track_file = trip_dir / f"{trip_dir.name}.track"

        if not track_file.exists():
            continue

        track = pd.read_csv(
            track_file,
            sep=r"\s+",
            header=None,
            names=["longitude", "latitude", "timestamp"]
        )

        if len(track) < 2:
            continue

        time_diff = track["timestamp"].diff().iloc[1:].to_numpy()

        distances = haversine_distance(
            track["longitude"].iloc[:-1].to_numpy(),
            track["latitude"].iloc[:-1].to_numpy(),
            track["longitude"].iloc[1:].to_numpy(),
            track["latitude"].iloc[1:].to_numpy()
        )

        valid_time = time_diff > 0

        speeds = np.full(len(time_diff), np.nan)
        speeds[valid_time] = (
            distances[valid_time] / time_diff[valid_time] * 3.6
        )

        results.append({
            "partition": partition,
            "trip_id": trip_dir.name,
            "total_intervals": len(time_diff),
            "intervals_over_1_sec": int((time_diff > 1).sum()),
            "intervals_over_5_sec": int((time_diff > 5).sum()),
            "intervals_over_10_sec": int((time_diff > 10).sum()),
            "zero_or_negative_intervals": int((time_diff <= 0).sum()),
            "maximum_gap_sec": float(np.max(time_diff)),
            "average_gap_sec": float(np.mean(time_diff)),
            "maximum_segment_speed_kmh": (
                float(np.nanmax(speeds))
                if np.isfinite(speeds).any() else np.nan
            )
        })

report = pd.DataFrame(results)

report.to_csv(
    OUTPUT_DIR / "gps_interval_quality.csv",
    index=False
)

print("========== GPS INTERVAL QUALITY ANALYSIS ==========")
print("Total trips analysed:", len(report))

print("\nTotal irregular intervals:")
print("Over 1 second:", report["intervals_over_1_sec"].sum())
print("Over 5 seconds:", report["intervals_over_5_sec"].sum())
print("Over 10 seconds:", report["intervals_over_10_sec"].sum())
print("Zero or negative:", report["zero_or_negative_intervals"].sum())

print("\nLargest timestamp gaps:")
print(
    report.nlargest(10, "maximum_gap_sec")[
        ["partition", "trip_id", "maximum_gap_sec"]
    ].to_string(index=False)
)

print("\nTrips with the highest segment speeds:")
print(
    report.nlargest(10, "maximum_segment_speed_kmh")[
        ["partition", "trip_id", "maximum_segment_speed_kmh"]
    ].to_string(index=False)
)

print("\nReport saved successfully!")


========== GPS INTERVAL QUALITY ANALYSIS ==========
Total trips analysed: 90

Total irregular intervals:
Over 1 second: 7550
Over 5 seconds: 1260
Over 10 seconds: 617
Zero or negative: 0

Largest timestamp gaps:
partition  trip_id  maximum_gap_sec
    train 00000090         153064.0
    train 00000078          52810.0
    train 00000069           2658.0
    train 00000021           2161.0
    train 00000011            909.0
    train 00000042            670.0
    train 00000081            536.0
    train 00000024            458.0
    train 00000044            341.0
    train 00000084            312.0

Trips with the highest segment speeds:
   partition  trip_id  maximum_segment_speed_kmh
       train 00000079                 648.496895
       train 00000033                 511.023287
       train 00000013                 509.554412
       train 00000043                 473.987835
       train 00000092                 305.973667
       train 00000057                 301.704892
       tr

## 08_inspect_gps_anomalies

Source: `08_inspect_gps_anomalies.py`

In [35]:

from pathlib import Path

import numpy as np
import pandas as pd

BASE_DIR = DATA_DIR
OUTPUT_DIR = BASE_DIR / "csv_output" / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EARTH_RADIUS = 6371000
GAP_THRESHOLD = 10
SPEED_THRESHOLD = 160

def haversine_distance(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(
        np.radians, [lon1, lat1, lon2, lat2]
    )

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )

    return 2 * EARTH_RADIUS * np.arcsin(np.sqrt(a))

anomalies = []

for partition in ["train", "student_test"]:
    partition_dir = BASE_DIR / partition

    for trip_dir in sorted(partition_dir.iterdir()):
        if not trip_dir.is_dir():
            continue

        track_file = trip_dir / f"{trip_dir.name}.track"

        if not track_file.exists():
            continue

        track = pd.read_csv(
            track_file,
            sep=r"\s+",
            header=None,
            names=["longitude", "latitude", "timestamp"]
        )

        if len(track) < 2:
            continue

        for i in range(1, len(track)):
            previous = track.iloc[i - 1]
            current = track.iloc[i]

            time_gap = current["timestamp"] - previous["timestamp"]

            distance = haversine_distance(
                previous["longitude"],
                previous["latitude"],
                current["longitude"],
                current["latitude"]
            )

            speed = (
                distance / time_gap * 3.6
                if time_gap > 0 else np.nan
            )

            reasons = []

            if time_gap > GAP_THRESHOLD:
                reasons.append("Large timestamp gap")

            if np.isfinite(speed) and speed > SPEED_THRESHOLD:
                reasons.append("High segment speed")

            if reasons:
                anomalies.append({
                    "partition": partition,
                    "trip_id": trip_dir.name,
                    "previous_point_index": i - 1,
                    "current_point_index": i,
                    "previous_timestamp": previous["timestamp"],
                    "current_timestamp": current["timestamp"],
                    "time_gap_sec": time_gap,
                    "distance_m": distance,
                    "speed_kmh": speed,
                    "previous_longitude": previous["longitude"],
                    "previous_latitude": previous["latitude"],
                    "current_longitude": current["longitude"],
                    "current_latitude": current["latitude"],
                    "reason": "; ".join(reasons)
                })

report = pd.DataFrame(anomalies)

output_file = OUTPUT_DIR / "gps_anomaly_details.csv"
report.to_csv(output_file, index=False)

print("========== GPS ANOMALY INSPECTION ==========")
print("Total suspicious segments:", len(report))

if not report.empty:
    print("\nAnomalies by reason:")
    print(report["reason"].value_counts())

    print("\nTop 15 highest-speed segments:")
    print(
        report.nlargest(15, "speed_kmh")[
            ["trip_id", "time_gap_sec", "distance_m", "speed_kmh", "reason"]
        ].to_string(index=False)
    )

    print("\nLargest timestamp gaps:")
    print(
        report.nlargest(10, "time_gap_sec")[
            ["trip_id", "previous_timestamp", "current_timestamp", "time_gap_sec"]
        ].to_string(index=False)
    )

print("\nDetailed report saved at:", output_file)


========== GPS ANOMALY INSPECTION ==========
Total suspicious segments: 3132

Anomalies by reason:
reason
High segment speed     2515
Large timestamp gap     617
Name: count, dtype: int64

Top 15 highest-speed segments:
 trip_id  time_gap_sec  distance_m  speed_kmh             reason
00000079           1.0  180.138026 648.496895 High segment speed
00000033           1.0  141.950913 511.023287 High segment speed
00000013           1.0  141.542892 509.554412 High segment speed
00000079           1.0  133.011830 478.842586 High segment speed
00000043           1.0  131.663287 473.987835 High segment speed
00000013           1.0  128.912768 464.085964 High segment speed
00000013           1.0  114.286466 411.431279 High segment speed
00000013           1.0  113.230590 407.630123 High segment speed
00000013           1.0  112.119139 403.628902 High segment speed
00000043           1.0  109.873234 395.543642 High segment speed
00000013           1.0  108.515746 390.656685 High segment speed


## 09_inspect_suspicious_trips

Source: `09_inspect_suspicious_trips.py`

In [36]:

from pathlib import Path

import pandas as pd

BASE_DIR = DATA_DIR
REPORT_DIR = BASE_DIR / "csv_output" / "reports"

anomalies = pd.read_csv(
    REPORT_DIR / "gps_anomaly_details.csv",
    dtype={"trip_id": str}
)

def inspect_trip(partition, trip_id, anomaly_type, top_n=3):
    trip_anomalies = anomalies[
        (anomalies["partition"] == partition) &
        (anomalies["trip_id"] == trip_id)
    ].copy()

    if anomaly_type == "gap":
        trip_anomalies = trip_anomalies[
            trip_anomalies["time_gap_sec"] > 10
        ].nlargest(top_n, "time_gap_sec")

    else:
        trip_anomalies = trip_anomalies[
            trip_anomalies["speed_kmh"] > 160
        ].nlargest(top_n, "speed_kmh")

    track_file = (
        BASE_DIR / partition / trip_id / f"{trip_id}.track"
    )

    track = pd.read_csv(
        track_file,
        sep=r"\s+",
        header=None,
        names=["longitude", "latitude", "timestamp"]
    )

    print(f"\n========== TRIP {trip_id} ==========")

    if trip_anomalies.empty:
        print("No matching suspicious segments found.")
        return

    for _, anomaly in trip_anomalies.iterrows():
        previous_index = int(anomaly["previous_point_index"])
        current_index = int(anomaly["current_point_index"])

        print("\nSuspicious segment:")
        print("Reason:", anomaly["reason"])
        print("Time gap:", anomaly["time_gap_sec"], "seconds")
        print("Distance:", round(anomaly["distance_m"], 2), "metres")
        print("Speed:", round(anomaly["speed_kmh"], 2), "km/h")

        start = max(0, previous_index - 2)
        end = min(len(track), current_index + 3)

        print("\nOriginal GPS observations:")
        print(track.iloc[start:end].to_string(index=True))

inspect_trip("train", "00000090", "gap", 3)
inspect_trip("train", "00000079", "speed", 3)
inspect_trip("train", "00000013", "speed", 3)



========== TRIP 00000090 ==========

Suspicious segment:
Reason: Large timestamp gap
Time gap: 153064.0 seconds
Distance: 3.3 metres
Speed: 0.0 km/h

Original GPS observations:
     longitude   latitude  timestamp
532  10.445810  51.621172      958.0
533  10.445839  51.621247      962.0
534  10.445741  51.621282      970.0
535  10.445779  51.621300   154034.0

Suspicious segment:
Reason: Large timestamp gap
Time gap: 18.0 seconds
Distance: 8.22 metres
Speed: 1.64 km/h

Original GPS observations:
     longitude   latitude  timestamp
381  10.444355  51.619742      515.0
382  10.444266  51.619733      517.0
383  10.444280  51.619603      520.0
384  10.444181  51.619562      538.0
385  10.443961  51.619608      540.0
386  10.443821  51.619627      542.0

Suspicious segment:
Reason: Large timestamp gap
Time gap: 13.0 seconds
Distance: 12.85 metres
Speed: 3.56 km/h

Original GPS observations:
     longitude   latitude  timestamp
432  10.444463  51.619365      657.0
433  10.444331  51.619187

## 10_gps_anomaly_summary

Source: `10_gps_anomaly_summary.py`

In [37]:

from pathlib import Path

import pandas as pd

BASE_DIR = DATA_DIR
REPORT_DIR = BASE_DIR / "csv_output" / "reports"

anomalies = pd.read_csv(
    REPORT_DIR / "gps_anomaly_details.csv",
    dtype={"trip_id": str}
)

summary = anomalies.groupby(
    ["partition", "trip_id"]
).agg(
    total_anomalies=("trip_id", "count"),
    large_gaps=("time_gap_sec", lambda x: (x > 10).sum()),
    high_speed_segments=("speed_kmh", lambda x: (x > 160).sum()),
    maximum_gap_sec=("time_gap_sec", "max"),
    maximum_speed_kmh=("speed_kmh", "max")
).reset_index()

summary = summary.sort_values(
    by="total_anomalies",
    ascending=False
)

output_file = REPORT_DIR / "gps_anomaly_summary.csv"

summary.to_csv(output_file, index=False)

print("========== GPS ANOMALY SUMMARY ==========")
print("Trips containing anomalies:", len(summary))

print("\nTotal large gaps:", summary["large_gaps"].sum())
print("Total high-speed segments:", summary["high_speed_segments"].sum())

print("\nTop 15 trips with the most anomalies:")
print(summary.head(15).to_string(index=False))

print("\nSummary saved successfully!")
print(output_file)


========== GPS ANOMALY SUMMARY ==========
Trips containing anomalies: 72

Total large gaps: 617
Total high-speed segments: 2515

Top 15 trips with the most anomalies:
   partition  trip_id  total_anomalies  large_gaps  high_speed_segments  maximum_gap_sec  maximum_speed_kmh
       train 00000032             1111           7                 1104             41.0         299.798693
       train 00000001              557           1                  556             16.0         260.716176
       train 00000004              215         215                    0             51.0          31.976572
       train 00000055              151           1                  150            184.0         235.887157
       train 00000013              141           0                  141              1.0         509.554412
       train 00000028              113           6                  107            196.0         243.079570
       train 00000024              108           2                  106      

## 11_gps_preprocessing

Source: `11_gps_preprocessing.py`

In [38]:

from pathlib import Path

import numpy as np
import pandas as pd

BASE_DIR = DATA_DIR
OUTPUT_DIR = BASE_DIR / "csv_output" / "processed"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EARTH_RADIUS = 6371000
GAP_THRESHOLD = 10
SPEED_THRESHOLD = 160

def haversine_distance(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(
        np.radians, [lon1, lat1, lon2, lat2]
    )

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )

    return 2 * EARTH_RADIUS * np.arcsin(np.sqrt(a))

processing_summary = []

for partition in ["train", "student_test"]:
    partition_dir = BASE_DIR / partition

    for trip_dir in sorted(partition_dir.iterdir()):
        if not trip_dir.is_dir():
            continue

        trip_id = trip_dir.name
        track_file = trip_dir / f"{trip_id}.track"

        if not track_file.exists():
            continue

        track = pd.read_csv(
            track_file,
            sep=r"\s+",
            header=None,
            names=["longitude", "latitude", "timestamp"]
        )

        track["time_gap_sec"] = track["timestamp"].diff()

        distances = np.full(len(track), np.nan)

        if len(track) > 1:
            distances[1:] = haversine_distance(
                track["longitude"].iloc[:-1].to_numpy(),
                track["latitude"].iloc[:-1].to_numpy(),
                track["longitude"].iloc[1:].to_numpy(),
                track["latitude"].iloc[1:].to_numpy()
            )

        track["distance_from_previous_m"] = distances

        track["speed_from_previous_kmh"] = (
            track["distance_from_previous_m"]
            / track["time_gap_sec"]
            * 3.6
        )

        track.loc[
            track["time_gap_sec"] <= 0,
            "speed_from_previous_kmh"
        ] = np.nan

        track["large_time_gap"] = (
            track["time_gap_sec"] > GAP_THRESHOLD
        )

        track["high_speed_segment"] = (
            track["speed_from_previous_kmh"] > SPEED_THRESHOLD
        )

        track["suspicious_segment"] = (
            track["large_time_gap"] |
            track["high_speed_segment"]
        )

        output_partition = OUTPUT_DIR / partition
        output_partition.mkdir(parents=True, exist_ok=True)

        output_file = output_partition / f"{trip_id}.csv"
        track.to_csv(output_file, index=False)

        processing_summary.append({
            "partition": partition,
            "trip_id": trip_id,
            "original_points": len(track),
            "large_time_gaps": int(track["large_time_gap"].sum()),
            "high_speed_segments": int(track["high_speed_segment"].sum()),
            "flagged_segments": int(track["suspicious_segment"].sum()),
            "processed_points": len(track)
        })

summary = pd.DataFrame(processing_summary)

summary.to_csv(
    BASE_DIR / "csv_output" / "reports" / "gps_preprocessing_summary.csv",
    index=False
)

print("========== GPS PREPROCESSING ==========")
print("Trips processed:", len(summary))
print("Original GPS points:", summary["original_points"].sum())
print("Processed GPS points:", summary["processed_points"].sum())
print("Large timestamp gaps:", summary["large_time_gaps"].sum())
print("High-speed segments:", summary["high_speed_segments"].sum())
print("Flagged segments:", summary["flagged_segments"].sum())
print("Original GPS observations preserved:", 
      (summary["original_points"] == summary["processed_points"]).all())

print("\nProcessed files saved in:", OUTPUT_DIR)
print("Preprocessing summary saved successfully!")


========== GPS PREPROCESSING ==========
Trips processed: 90
Original GPS points: 227577
Processed GPS points: 227577
Large timestamp gaps: 617
High-speed segments: 2515
Flagged segments: 3132
Original GPS observations preserved: True

Processed files saved in: C:\Users\vagee\OneDrive\Documents\ML HACKATHON\ML_Hackathon\data\csv_output\processed
Preprocessing summary saved successfully!


## 12_validate_processed_data

Source: `12_validate_processed_data.py`

In [39]:

from pathlib import Path

import numpy as np
import pandas as pd

BASE_DIR = DATA_DIR
PROCESSED_DIR = BASE_DIR / "csv_output" / "processed"

results = []

for partition in ["train", "student_test"]:
    original_dir = BASE_DIR / partition
    processed_dir = PROCESSED_DIR / partition

    for trip_dir in sorted(original_dir.iterdir()):
        if not trip_dir.is_dir():
            continue

        trip_id = trip_dir.name

        original_file = trip_dir / f"{trip_id}.track"
        processed_file = processed_dir / f"{trip_id}.csv"

        if not original_file.exists() or not processed_file.exists():
            results.append({
                "partition": partition,
                "trip_id": trip_id,
                "status": "MISSING FILE"
            })
            continue

        original = pd.read_csv(
            original_file,
            sep=r"\s+",
            header=None,
            names=["longitude", "latitude", "timestamp"]
        )

        processed = pd.read_csv(processed_file)

        coordinates_match = np.allclose(
            original[["longitude", "latitude", "timestamp"]].to_numpy(),
            processed[["longitude", "latitude", "timestamp"]].to_numpy(),
            equal_nan=True
        )

        valid_coordinates = (
            processed["longitude"].between(-180, 180).all()
            and processed["latitude"].between(-90, 90).all()
        )

        flags_match = (
            processed["large_time_gap"].fillna(False)
            == (processed["time_gap_sec"] > 10)
        ).all() and (
            processed["high_speed_segment"].fillna(False)
            == (processed["speed_from_previous_kmh"] > 160)
        ).all()

        passed = (
            len(original) == len(processed)
            and coordinates_match
            and valid_coordinates
            and flags_match
        )

        results.append({
            "partition": partition,
            "trip_id": trip_id,
            "original_points": len(original),
            "processed_points": len(processed),
            "coordinates_and_timestamps_match": coordinates_match,
            "valid_coordinates": valid_coordinates,
            "flags_match": flags_match,
            "status": "PASS" if passed else "FAIL"
        })

report = pd.DataFrame(results)

output_file = BASE_DIR / "csv_output" / "reports" / "processed_data_validation.csv"
output_file.parent.mkdir(parents=True, exist_ok=True)
report.to_csv(output_file, index=False)

print("========== PROCESSED DATA VALIDATION ==========")
print("Total trips checked:", len(report))
print("\nValidation results:")
print(report["status"].value_counts())

print("\nFailed trips:")
print(report[report["status"] != "PASS"].to_string(index=False))

print("\nAll trips passed:", (report["status"] == "PASS").all())
print("Validation report saved:", output_file)


========== PROCESSED DATA VALIDATION ==========
Total trips checked: 90

Validation results:
status
PASS    90
Name: count, dtype: int64

Failed trips:
Empty DataFrame
Columns: [partition, trip_id, original_points, processed_points, coordinates_and_timestamps_match, valid_coordinates, flags_match, status]
Index: []

All trips passed: True
Validation report saved: C:\Users\vagee\OneDrive\Documents\ML HACKATHON\ML_Hackathon\data\csv_output\reports\processed_data_validation.csv


## 13_road_network_preprocessing

Source: `13_road_network_preprocessing.py`

In [40]:

from pathlib import Path

import numpy as np
import pandas as pd

BASE_DIR = DATA_DIR
TRIP_ID = "00000001"

TRIP_DIR = BASE_DIR / "train" / TRIP_ID
OUTPUT_DIR = BASE_DIR / "csv_output" / "processed" / "road_network"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EARTH_RADIUS = 6371000

nodes = pd.read_csv(
    TRIP_DIR / f"{TRIP_ID}.nodes",
    sep=r"\s+",
    header=None,
    names=["longitude", "latitude"]
)

arcs = pd.read_csv(
    TRIP_DIR / f"{TRIP_ID}.arcs",
    sep=r"\s+",
    header=None,
    names=["from_node_id", "to_node_id"]
)

arcs["arc_id"] = np.arange(len(arcs))

from_nodes = nodes.iloc[arcs["from_node_id"]].reset_index(drop=True)
to_nodes = nodes.iloc[arcs["to_node_id"]].reset_index(drop=True)

lon1 = np.radians(from_nodes["longitude"].to_numpy())
lat1 = np.radians(from_nodes["latitude"].to_numpy())
lon2 = np.radians(to_nodes["longitude"].to_numpy())
lat2 = np.radians(to_nodes["latitude"].to_numpy())

dlon = lon2 - lon1
dlat = lat2 - lat1

a = (
    np.sin(dlat / 2) ** 2
    + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
)

distances = 2 * EARTH_RADIUS * np.arcsin(np.sqrt(a))

arcs["length_m"] = distances

arcs["self_loop"] = (
    arcs["from_node_id"] == arcs["to_node_id"]
)

arcs["duplicate_directed_arc"] = arcs.duplicated(
    subset=["from_node_id", "to_node_id"],
    keep="first"
)

arcs["valid_length"] = (
    np.isfinite(arcs["length_m"])
    & (arcs["length_m"] > 0)
)

output_file = OUTPUT_DIR / f"{TRIP_ID}_processed_arcs.csv"
arcs.to_csv(output_file, index=False)

print("========== ROAD NETWORK PREPROCESSING ==========")
print("Trip ID:", TRIP_ID)
print("Total nodes:", len(nodes))
print("Total arcs:", len(arcs))
print("Self-loops:", int(arcs["self_loop"].sum()))
print("Duplicate directed arcs:", int(arcs["duplicate_directed_arc"].sum()))
print("Valid arc lengths:", int(arcs["valid_length"].sum()))
print("Invalid arc lengths:", int((~arcs["valid_length"]).sum()))

print("\nArc length statistics (metres):")
print(arcs["length_m"].describe().round(3))

print("\nFirst 10 processed arcs:")
print(arcs.head(10).to_string(index=False))

print("\nProcessed road network saved at:")
print(output_file)


========== ROAD NETWORK PREPROCESSING ==========
Trip ID: 00000001
Total nodes: 88809
Total arcs: 165790
Self-loops: 0
Duplicate directed arcs: 292
Valid arc lengths: 165782
Invalid arc lengths: 8

Arc length statistics (metres):
count    165790.000
mean         39.627
std          55.359
min           0.000
25%           9.269
50%          20.198
75%          46.751
max        1106.457
Name: length_m, dtype: float64

First 10 processed arcs:
 from_node_id  to_node_id  arc_id   length_m  self_loop  duplicate_directed_arc  valid_length
            0         281       0  78.194330      False                   False          True
            0           1       1 217.008757      False                   False          True
            0           5       2 103.432653      False                   False          True
            1           0       3 217.008757      False                   False          True
            1       40750       4  45.229985      False                   False    

## 14_road_network_quality_analysis

Source: `14_road_network_quality_analysis.py`

In [41]:

from pathlib import Path

import numpy as np
import pandas as pd

BASE_DIR = DATA_DIR
TRIP_ID = "00000001"

INPUT_DIR = BASE_DIR / "csv_output" / "processed" / "road_network"
OUTPUT_DIR = BASE_DIR / "csv_output" / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

arcs = pd.read_csv(
    INPUT_DIR / f"{TRIP_ID}_processed_arcs.csv"
)

nodes = pd.read_csv(
    BASE_DIR / "train" / TRIP_ID / f"{TRIP_ID}.nodes",
    sep=r"\s+",
    header=None,
    names=["longitude", "latitude"]
)

invalid_arcs = arcs[
    (~arcs["valid_length"]) |
    (~np.isfinite(arcs["length_m"]))
]

duplicate_arcs = arcs[
    arcs.duplicated(
        subset=["from_node_id", "to_node_id"],
        keep=False
    )
].sort_values(["from_node_id", "to_node_id"])

long_arcs = arcs.nlargest(20, "length_m")

print("========== ROAD NETWORK QUALITY ANALYSIS ==========")

print("\n1. INVALID ARCS")
print("Total:", len(invalid_arcs))
print(invalid_arcs.to_string(index=False))

print("\n2. DUPLICATE DIRECTED ARCS")
print("Total rows involved:", len(duplicate_arcs))
print(duplicate_arcs.head(20).to_string(index=False))

print("\n3. LONGEST ROAD ARCS")
print(long_arcs[
    ["arc_id", "from_node_id", "to_node_id", "length_m"]
].to_string(index=False))

print("\n4. ZERO-LENGTH ARCS")
zero_arcs = arcs[arcs["length_m"] == 0]
print("Total:", len(zero_arcs))
print(zero_arcs.to_string(index=False))

invalid_arcs.to_csv(
    OUTPUT_DIR / "invalid_road_arcs.csv",
    index=False
)

duplicate_arcs.to_csv(
    OUTPUT_DIR / "duplicate_road_arcs.csv",
    index=False
)

long_arcs.to_csv(
    OUTPUT_DIR / "long_road_arcs.csv",
    index=False
)

print("\nReports saved successfully!")


========== ROAD NETWORK QUALITY ANALYSIS ==========

1. INVALID ARCS
Total: 8
 from_node_id  to_node_id  arc_id  length_m  self_loop  duplicate_directed_arc  valid_length
        25407       72348   51615       0.0      False                   False         False
        33761       72024   67864       0.0      False                   False         False
        45593       45594   89836       0.0      False                   False         False
        45594       45593   89839       0.0      False                   False         False
        72348       25407  136226       0.0      False                   False         False
        78592       85421  147014       0.0      False                   False         False
        85421       78592  159604       0.0      False                   False         False
        85558       32380  159841       0.0      False                   False         False

2. DUPLICATE DIRECTED ARCS
Total rows involved: 584
 from_node_id  to_node_id  arc_i

## 15_inspect_road_network_anomalies

Source: `15_inspect_road_network_anomalies.py`

In [42]:

from pathlib import Path

import pandas as pd

BASE_DIR = DATA_DIR
TRIP_ID = "00000001"

TRIP_DIR = BASE_DIR / "train" / TRIP_ID
INPUT_DIR = BASE_DIR / "csv_output" / "processed" / "road_network"
REPORT_DIR = BASE_DIR / "csv_output" / "reports"

arcs = pd.read_csv(
    INPUT_DIR / f"{TRIP_ID}_processed_arcs.csv"
)

nodes = pd.read_csv(
    TRIP_DIR / f"{TRIP_ID}.nodes",
    sep=r"\s+",
    header=None,
    names=["longitude", "latitude"]
)

invalid_arcs = arcs[arcs["length_m"] == 0]

duplicate_arcs = arcs[
    arcs.duplicated(
        subset=["from_node_id", "to_node_id"],
        keep=False
    )
]

print("========== ZERO-LENGTH ARC INSPECTION ==========")

for _, arc in invalid_arcs.iterrows():
    start_id = int(arc["from_node_id"])
    end_id = int(arc["to_node_id"])

    start_node = nodes.iloc[start_id]
    end_node = nodes.iloc[end_id]

    print(f"\nArc ID: {int(arc['arc_id'])}")
    print("Start node:", start_id)
    print("End node:", end_id)
    print("Start coordinates:", tuple(start_node))
    print("End coordinates:", tuple(end_node))
    print(
        "Coordinates identical:",
        start_node.equals(end_node)
    )

print("\n========== DUPLICATE ARC INSPECTION ==========")

duplicate_groups = duplicate_arcs.groupby(
    ["from_node_id", "to_node_id"]
)

print("Total duplicate groups:", len(duplicate_groups))

print("\nFirst 15 duplicate groups:")

for (from_id, to_id), group in list(duplicate_groups)[:15]:
    print({
        "from_node": from_id,
        "to_node": to_id,
        "arc_ids": group["arc_id"].tolist(),
        "lengths": group["length_m"].round(3).tolist()
    })

print("\n========== INSPECTION COMPLETED ==========")


========== ZERO-LENGTH ARC INSPECTION ==========

Arc ID: 51615
Start node: 25407
End node: 72348
Start coordinates: (8.616805, 49.994635)
End coordinates: (8.616805, 49.994635)
Coordinates identical: True

Arc ID: 67864
Start node: 33761
End node: 72024
Start coordinates: (8.598857, 50.135592)
End coordinates: (8.598857, 50.135592)
Coordinates identical: True

Arc ID: 89836
Start node: 45593
End node: 45594
Start coordinates: (8.57204, 50.168436)
End coordinates: (8.57204, 50.168436)
Coordinates identical: True

Arc ID: 89839
Start node: 45594
End node: 45593
Start coordinates: (8.57204, 50.168436)
End coordinates: (8.57204, 50.168436)
Coordinates identical: True

Arc ID: 136226
Start node: 72348
End node: 25407
Start coordinates: (8.616805, 49.994635)
End coordinates: (8.616805, 49.994635)
Coordinates identical: True

Arc ID: 147014
Start node: 78592
End node: 85421
Start coordinates: (8.599377, 49.929581)
End coordinates: (8.599377, 49.929581)
Coordinates identical: True

Arc ID: 15

## 16_process_all_road_networks

Source: `16_process_all_road_networks.py`

In [43]:

from pathlib import Path

import numpy as np
import pandas as pd

BASE_DIR = DATA_DIR
OUTPUT_DIR = BASE_DIR / "csv_output" / "processed" / "road_network"

EARTH_RADIUS = 6371000

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def haversine_distance(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(
        np.radians, [lon1, lat1, lon2, lat2]
    )

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )

    return 2 * EARTH_RADIUS * np.arcsin(np.sqrt(a))

summary = []

for partition in ["train", "student_test"]:
    partition_dir = BASE_DIR / partition

    for trip_dir in sorted(partition_dir.iterdir()):
        if not trip_dir.is_dir():
            continue

        trip_id = trip_dir.name

        nodes_file = trip_dir / f"{trip_id}.nodes"
        arcs_file = trip_dir / f"{trip_id}.arcs"

        if not nodes_file.exists() or not arcs_file.exists():
            continue

        nodes = pd.read_csv(
            nodes_file,
            sep=r"\s+",
            header=None,
            names=["longitude", "latitude"]
        )

        arcs = pd.read_csv(
            arcs_file,
            sep=r"\s+",
            header=None,
            names=["from_node_id", "to_node_id"]
        )

        arcs["arc_id"] = np.arange(len(arcs))

        valid_connections = (
            arcs["from_node_id"].between(0, len(nodes) - 1)
            & arcs["to_node_id"].between(0, len(nodes) - 1)
        )

        arcs["valid_connection"] = valid_connections

        lengths = np.full(len(arcs), np.nan)

        valid_arcs = arcs.loc[valid_connections]

        if len(valid_arcs) > 0:
            from_nodes = nodes.iloc[
                valid_arcs["from_node_id"].to_numpy()
            ]

            to_nodes = nodes.iloc[
                valid_arcs["to_node_id"].to_numpy()
            ]

            lengths[valid_connections.to_numpy()] = haversine_distance(
                from_nodes["longitude"].to_numpy(),
                from_nodes["latitude"].to_numpy(),
                to_nodes["longitude"].to_numpy(),
                to_nodes["latitude"].to_numpy()
            )

        arcs["length_m"] = lengths

        arcs["self_loop"] = (
            arcs["from_node_id"] == arcs["to_node_id"]
        )

        arcs["duplicate_directed_arc"] = arcs.duplicated(
            subset=["from_node_id", "to_node_id"],
            keep="first"
        )

        arcs["valid_length"] = (
            np.isfinite(arcs["length_m"])
            & (arcs["length_m"] > 0)
            & arcs["valid_connection"]
        )

        trip_output = OUTPUT_DIR / partition / trip_id
        trip_output.mkdir(parents=True, exist_ok=True)

        nodes_output = nodes.copy()
        nodes_output.insert(0, "node_id", np.arange(len(nodes)))

        nodes_output.to_csv(
            trip_output / f"{trip_id}_processed_nodes.csv",
            index=False
        )

        arcs.to_csv(
            trip_output / f"{trip_id}_processed_arcs.csv",
            index=False
        )

        summary.append({
            "partition": partition,
            "trip_id": trip_id,
            "total_nodes": len(nodes),
            "total_arcs": len(arcs),
            "invalid_connections": int((~valid_connections).sum()),
            "self_loops": int(arcs["self_loop"].sum()),
            "duplicate_directed_arcs": int(
                arcs["duplicate_directed_arc"].sum()
            ),
            "zero_length_arcs": int(
                (arcs["length_m"] == 0).sum()
            ),
            "valid_arcs": int(arcs["valid_length"].sum())
        })

summary_df = pd.DataFrame(summary)

REPORT_DIR = BASE_DIR / "csv_output" / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

summary_df.to_csv(
    REPORT_DIR / "all_road_networks_summary.csv",
    index=False
)

print("========== ALL ROAD NETWORKS PREPROCESSING ==========")
print("Trips processed:", len(summary_df))
print("Training trips:", (summary_df["partition"] == "train").sum())
print("Student testing trips:", (summary_df["partition"] == "student_test").sum())
print("Total nodes:", summary_df["total_nodes"].sum())
print("Total arcs:", summary_df["total_arcs"].sum())
print("Invalid connections:", summary_df["invalid_connections"].sum())
print("Self-loops:", summary_df["self_loops"].sum())
print("Duplicate directed arcs:", summary_df["duplicate_directed_arcs"].sum())
print("Zero-length arcs:", summary_df["zero_length_arcs"].sum())
print("Valid arcs:", summary_df["valid_arcs"].sum())

print("\nSummary saved successfully!")
print("Processed road networks saved at:", OUTPUT_DIR)


========== ALL ROAD NETWORKS PREPROCESSING ==========
Trips processed: 90
Training trips: 80
Student testing trips: 10
Total nodes: 2024174
Total arcs: 4061945
Invalid connections: 0
Self-loops: 24
Duplicate directed arcs: 1903
Zero-length arcs: 106
Valid arcs: 4061839

Summary saved successfully!
Processed road networks saved at: C:\Users\vagee\OneDrive\Documents\ML HACKATHON\ML_Hackathon\data\csv_output\processed\road_network


## 17_validate_all_road_networks

Source: `17_validate_all_road_networks.py`

In [44]:

from pathlib import Path

import numpy as np
import pandas as pd

BASE_DIR = DATA_DIR
PROCESSED_DIR = BASE_DIR / "csv_output" / "processed" / "road_network"

results = []

for partition in ["train", "student_test"]:
    partition_dir = BASE_DIR / partition

    for trip_dir in sorted(partition_dir.iterdir()):
        if not trip_dir.is_dir():
            continue

        trip_id = trip_dir.name

        original_nodes_file = trip_dir / f"{trip_id}.nodes"
        original_arcs_file = trip_dir / f"{trip_id}.arcs"

        processed_dir = PROCESSED_DIR / partition / trip_id

        processed_nodes_file = (
            processed_dir / f"{trip_id}_processed_nodes.csv"
        )

        processed_arcs_file = (
            processed_dir / f"{trip_id}_processed_arcs.csv"
        )

        if not processed_nodes_file.exists() or not processed_arcs_file.exists():
            results.append({
                "partition": partition,
                "trip_id": trip_id,
                "status": "MISSING FILE"
            })
            continue

        original_nodes = pd.read_csv(
            original_nodes_file,
            sep=r"\s+",
            header=None,
            names=["longitude", "latitude"]
        )

        original_arcs = pd.read_csv(
            original_arcs_file,
            sep=r"\s+",
            header=None,
            names=["from_node_id", "to_node_id"]
        )

        processed_nodes = pd.read_csv(processed_nodes_file)
        processed_arcs = pd.read_csv(processed_arcs_file)

        node_count_match = len(original_nodes) == len(processed_nodes)
        arc_count_match = len(original_arcs) == len(processed_arcs)

        node_ids_match = np.array_equal(
            processed_nodes["node_id"].to_numpy(),
            np.arange(len(original_nodes))
        )

        node_coordinates_match = np.allclose(
            original_nodes[["longitude", "latitude"]].to_numpy(),
            processed_nodes[["longitude", "latitude"]].to_numpy()
        )

        arc_ids_match = np.array_equal(
            processed_arcs["arc_id"].to_numpy(),
            np.arange(len(original_arcs))
        )

        connections_match = np.array_equal(
            original_arcs[["from_node_id", "to_node_id"]].to_numpy(),
            processed_arcs[["from_node_id", "to_node_id"]].to_numpy()
        )

        lengths_valid = (
            np.isfinite(processed_arcs["length_m"]).all()
            and (processed_arcs["length_m"] >= 0).all()
        )

        passed = all([
            node_count_match,
            arc_count_match,
            node_ids_match,
            node_coordinates_match,
            arc_ids_match,
            connections_match,
            lengths_valid
        ])

        results.append({
            "partition": partition,
            "trip_id": trip_id,
            "node_count_match": node_count_match,
            "arc_count_match": arc_count_match,
            "node_ids_match": node_ids_match,
            "node_coordinates_match": node_coordinates_match,
            "arc_ids_match": arc_ids_match,
            "connections_match": connections_match,
            "lengths_valid": lengths_valid,
            "status": "PASS" if passed else "FAIL"
        })

report = pd.DataFrame(results)

report_dir = BASE_DIR / "csv_output" / "reports"
report_dir.mkdir(parents=True, exist_ok=True)

report.to_csv(
    report_dir / "road_network_validation.csv",
    index=False
)

print("========== ROAD NETWORK VALIDATION ==========")
print("Total trips checked:", len(report))
print("\nValidation results:")
print(report["status"].value_counts())

print("\nFailed trips:")
print(report[report["status"] != "PASS"].to_string(index=False))

print("\nAll trips passed:", (report["status"] == "PASS").all())
print("Validation report saved successfully!")


========== ROAD NETWORK VALIDATION ==========
Total trips checked: 90

Validation results:
status
PASS    90
Name: count, dtype: int64

Failed trips:
Empty DataFrame
Columns: [partition, trip_id, node_count_match, arc_count_match, node_ids_match, node_coordinates_match, arc_ids_match, connections_match, lengths_valid, status]
Index: []

All trips passed: True
Validation report saved successfully!


## 18_validate_reference_routes

Source: `18_validate_reference_routes.py`

In [45]:

from pathlib import Path

import numpy as np
import pandas as pd

BASE_DIR = DATA_DIR
PROCESSED_DIR = BASE_DIR / "csv_output" / "processed" / "road_network"
REPORT_DIR = BASE_DIR / "csv_output" / "reports"

REPORT_DIR.mkdir(parents=True, exist_ok=True)

results = []

for partition in ["train", "student_test"]:
    partition_dir = BASE_DIR / partition

    for trip_dir in sorted(partition_dir.iterdir()):
        if not trip_dir.is_dir():
            continue

        trip_id = trip_dir.name

        route_file = trip_dir / f"{trip_id}.route"
        original_arcs_file = trip_dir / f"{trip_id}.arcs"

        processed_arcs_file = (
            PROCESSED_DIR / partition / trip_id /
            f"{trip_id}_processed_arcs.csv"
        )

        if not all([
            route_file.exists(),
            original_arcs_file.exists(),
            processed_arcs_file.exists()
        ]):
            results.append({
                "partition": partition,
                "trip_id": trip_id,
                "status": "MISSING FILE"
            })
            continue

        route = pd.read_csv(
            route_file,
            sep=r"\s+",
            header=None,
            names=["arc_id"]
        )

        original_arcs = pd.read_csv(
            original_arcs_file,
            sep=r"\s+",
            header=None,
            names=["from_node_id", "to_node_id"]
        )

        processed_arcs = pd.read_csv(processed_arcs_file)

        route_ids = route["arc_id"].to_numpy()

        valid_ids = (
            (route_ids >= 0) &
            (route_ids < len(original_arcs))
        )

        route_ids_valid = bool(valid_ids.all())

        route_sequence_preserved = (
            len(route) > 0
            and np.array_equal(
                route_ids,
                route["arc_id"].to_numpy()
            )
        )

        if route_ids_valid:
            route_connections = original_arcs.iloc[
                route_ids
            ].reset_index(drop=True)

            if len(route_connections) > 1:
                transitions_connected = (
                    route_connections["to_node_id"].iloc[:-1].to_numpy()
                    ==
                    route_connections["from_node_id"].iloc[1:].to_numpy()
                )

                disconnected_transitions = int(
                    (~transitions_connected).sum()
                )
            else:
                disconnected_transitions = 0
        else:
            disconnected_transitions = np.nan

        processed_arc_ids = set(
            processed_arcs["arc_id"].to_numpy()
        )

        processed_ids_exist = all(
            int(arc_id) in processed_arc_ids
            for arc_id in route_ids
        )

        passed = (
            route_ids_valid
            and route_sequence_preserved
            and processed_ids_exist
        )

        results.append({
            "partition": partition,
            "trip_id": trip_id,
            "route_entries": len(route),
            "invalid_route_ids": int((~valid_ids).sum()),
            "route_sequence_preserved": route_sequence_preserved,
            "processed_arc_ids_exist": processed_ids_exist,
            "disconnected_transitions": disconnected_transitions,
            "status": "PASS" if passed else "FAIL"
        })

report = pd.DataFrame(results)

report.to_csv(
    REPORT_DIR / "reference_route_validation.csv",
    index=False
)

print("========== REFERENCE ROUTE VALIDATION ==========")
print("Total trips checked:", len(report))

print("\nValidation results:")
print(report["status"].value_counts())

print("\nTotal route entries:", 
      report["route_entries"].sum())

print("Invalid route IDs:",
      report["invalid_route_ids"].sum())

print("Disconnected consecutive transitions:",
      report["disconnected_transitions"].sum())

print("\nFailed trips:")
print(report[report["status"] != "PASS"].to_string(index=False))

print("\nAll trips passed:", (report["status"] == "PASS").all())

print("\nValidation report saved:")
print(REPORT_DIR / "reference_route_validation.csv")


========== REFERENCE ROUTE VALIDATION ==========
Total trips checked: 90

Validation results:
status
PASS    90
Name: count, dtype: int64

Total route entries: 34813
Invalid route IDs: 0
Disconnected consecutive transitions: 14

Failed trips:
Empty DataFrame
Columns: [partition, trip_id, route_entries, invalid_route_ids, route_sequence_preserved, processed_arc_ids_exist, disconnected_transitions, status]
Index: []

All trips passed: True

Validation report saved:
C:\Users\vagee\OneDrive\Documents\ML HACKATHON\ML_Hackathon\data\csv_output\reports\reference_route_validation.csv


## 19_inspect_disconnected_routes

Source: `19_inspect_disconnected_routes.py`

In [46]:

from pathlib import Path

import pandas as pd

BASE_DIR = DATA_DIR
REPORT_DIR = BASE_DIR / "csv_output" / "reports"

validation = pd.read_csv(
    REPORT_DIR / "reference_route_validation.csv",
    dtype={"trip_id": str}
)

disconnected = validation[
    validation["disconnected_transitions"] > 0
]

results = []

for _, row in disconnected.iterrows():
    partition = row["partition"]
    trip_id = row["trip_id"]

    trip_dir = BASE_DIR / partition / trip_id

    route = pd.read_csv(
        trip_dir / f"{trip_id}.route",
        sep=r"\s+",
        header=None,
        names=["arc_id"]
    )

    arcs = pd.read_csv(
        trip_dir / f"{trip_id}.arcs",
        sep=r"\s+",
        header=None,
        names=["from_node_id", "to_node_id"]
    )

    route_connections = arcs.iloc[
        route["arc_id"].to_numpy()
    ].reset_index(drop=True)

    for i in range(len(route_connections) - 1):
        current_to = int(
            route_connections.iloc[i]["to_node_id"]
        )

        next_from = int(
            route_connections.iloc[i + 1]["from_node_id"]
        )

        if current_to != next_from:
            results.append({
                "partition": partition,
                "trip_id": trip_id,
                "transition_index": i,
                "current_arc_id": int(route.iloc[i]["arc_id"]),
                "next_arc_id": int(route.iloc[i + 1]["arc_id"]),
                "current_to_node": current_to,
                "next_from_node": next_from
            })

details = pd.DataFrame(results)

details.to_csv(
    REPORT_DIR / "disconnected_route_details.csv",
    index=False
)

print("========== DISCONNECTED ROUTE INSPECTION ==========")
print("Affected trips:", len(disconnected))
print("Disconnected transitions:", len(details))

if not details.empty:
    print("\nDisconnected transition details:")
    print(details.to_string(index=False))

print("\nReport saved successfully!")


========== DISCONNECTED ROUTE INSPECTION ==========
Affected trips: 11
Disconnected transitions: 14

Disconnected transition details:
   partition  trip_id  transition_index  current_arc_id  next_arc_id  current_to_node  next_from_node
       train 00000002                53            1848        10863             5059            5289
       train 00000019                94           85589        51027              849           23488
       train 00000019               151           55059        55050            25413           25408
       train 00000019               300             953        23709            10107           10862
       train 00000033               164            1003         1002             1283             563
       train 00000041                12            1264          519              349             348
       train 00000042              1212           14661        17989             1762            8726
       train 00000069               153           

## 20_inspect_route_disconnections

Source: `20_inspect_route_disconnections.py`

In [47]:

from pathlib import Path

import numpy as np
import pandas as pd

BASE_DIR = DATA_DIR
REPORT_DIR = BASE_DIR / "csv_output" / "reports"

details = pd.read_csv(
    REPORT_DIR / "disconnected_route_details.csv",
    dtype={"trip_id": str}
)

EARTH_RADIUS = 6371000

def haversine(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(
        np.radians, [lon1, lat1, lon2, lat2]
    )

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )

    return 2 * EARTH_RADIUS * np.arcsin(np.sqrt(a))

results = []

for _, item in details.iterrows():
    partition = item["partition"]
    trip_id = item["trip_id"]

    trip_dir = BASE_DIR / partition / trip_id

    nodes = pd.read_csv(
        trip_dir / f"{trip_id}.nodes",
        sep=r"\s+",
        header=None,
        names=["longitude", "latitude"]
    )

    current_node = nodes.iloc[int(item["current_to_node"])]
    next_node = nodes.iloc[int(item["next_from_node"])]

    distance = haversine(
        current_node["longitude"],
        current_node["latitude"],
        next_node["longitude"],
        next_node["latitude"]
    )

    results.append({
        "partition": partition,
        "trip_id": trip_id,
        "current_arc_id": item["current_arc_id"],
        "next_arc_id": item["next_arc_id"],
        "current_to_node": item["current_to_node"],
        "next_from_node": item["next_from_node"],
        "distance_between_nodes_m": distance
    })

report = pd.DataFrame(results)

report.to_csv(
    REPORT_DIR / "route_disconnection_distances.csv",
    index=False
)

print("========== ROUTE DISCONNECTION DISTANCE ANALYSIS ==========")
print("Total disconnected transitions:", len(report))

print("\nDistance statistics:")
print(report["distance_between_nodes_m"].describe().round(3))

print("\nDetailed results:")
print(report.to_string(index=False))

print("\nReport saved successfully!")


========== ROUTE DISCONNECTION DISTANCE ANALYSIS ==========
Total disconnected transitions: 14

Distance statistics:
count     14.000
mean     140.786
std      163.655
min       10.620
25%       33.259
50%       82.490
75%      207.719
max      595.813
Name: distance_between_nodes_m, dtype: float64

Detailed results:
   partition  trip_id  current_arc_id  next_arc_id  current_to_node  next_from_node  distance_between_nodes_m
       train 00000002            1848        10863             5059            5289                324.678550
       train 00000019           85589        51027              849           23488                241.017356
       train 00000019           55059        55050            25413           25408                255.186061
       train 00000019             953        23709            10107           10862                595.813277
       train 00000033            1003         1002             1283             563                 91.934200
       train 00000041

## 21_final_data_processing_validation

Source: `21_final_data_processing_validation.py`

In [48]:

from pathlib import Path
import pandas as pd

BASE_DIR = DATA_DIR

PARTITIONS = {
    "train": BASE_DIR / "train",
    "student_test": BASE_DIR / "student_test"
}

PROCESSED_GPS = BASE_DIR / "csv_output" / "processed"
PROCESSED_ROAD = PROCESSED_GPS / "road_network"
REPORTS = BASE_DIR / "csv_output" / "reports"

REQUIRED_REPORTS = [
    "dataset_inventory.csv",
    "gps_gap_analysis.csv",
    "gps_anomaly_details.csv",
    "gps_preprocessing_summary.csv",
    "processed_data_validation.csv",
    "all_road_networks_summary.csv",
    "road_network_validation.csv",
    "reference_route_validation.csv",
    "disconnected_route_details.csv",
    "route_disconnection_distances.csv"
]

errors = []
total_trips = 0
gps_pass = 0
nodes_pass = 0
arcs_pass = 0

print("=" * 65)
print("FINAL DATA PROCESSING VALIDATION")
print("=" * 65)

# --------------------------------------------------
# 1. CHECK RAW AND PROCESSED TRIP FILES
# --------------------------------------------------

for partition, raw_folder in PARTITIONS.items():

    print(f"\nChecking partition: {partition}")

    if not raw_folder.exists():
        errors.append(f"Missing raw folder: {raw_folder}")
        print("[FAIL] Raw folder not found")
        continue

    # Search recursively inside subfolders
    trip_files = sorted(raw_folder.rglob("*.track"))

    print("Raw trips found:", len(trip_files))

    for track_file in trip_files:

        trip_id = track_file.stem
        total_trips += 1

        gps_file = (
            PROCESSED_GPS / partition / f"{trip_id}.csv"
        )

        road_folder = (
            PROCESSED_ROAD / partition / trip_id
        )

        nodes_file = (
            road_folder / f"{trip_id}_processed_nodes.csv"
        )

        arcs_file = (
            road_folder / f"{trip_id}_processed_arcs.csv"
        )

        # Check processed GPS
        try:
            if gps_file.exists():
                df = pd.read_csv(gps_file)

                if not df.empty:
                    gps_pass += 1
                else:
                    errors.append(f"Empty GPS file: {gps_file}")
            else:
                errors.append(f"Missing GPS file: {gps_file}")

        except Exception as e:
            errors.append(
                f"Unreadable GPS file: {gps_file} - {e}"
            )

        # Check processed nodes
        try:
            if nodes_file.exists():
                df = pd.read_csv(nodes_file)

                if not df.empty:
                    nodes_pass += 1
                else:
                    errors.append(f"Empty nodes file: {nodes_file}")
            else:
                errors.append(f"Missing nodes file: {nodes_file}")

        except Exception as e:
            errors.append(
                f"Unreadable nodes file: {nodes_file} - {e}"
            )

        # Check processed arcs
        try:
            if arcs_file.exists():
                df = pd.read_csv(arcs_file)

                if not df.empty:
                    arcs_pass += 1
                else:
                    errors.append(f"Empty arcs file: {arcs_file}")
            else:
                errors.append(f"Missing arcs file: {arcs_file}")

        except Exception as e:
            errors.append(
                f"Unreadable arcs file: {arcs_file} - {e}"
            )


# --------------------------------------------------
# 2. PRINT PROCESSED FILE SUMMARY
# --------------------------------------------------

print("\n" + "=" * 65)
print("PROCESSED FILE SUMMARY")
print("=" * 65)

print("Total trips:", total_trips)
print(f"GPS files:   {gps_pass}/{total_trips}")
print(f"Node files:  {nodes_pass}/{total_trips}")
print(f"Arc files:   {arcs_pass}/{total_trips}")

if total_trips == 0:
    errors.append(
        "No raw trips found. Check train and student_test folder paths."
    )


# --------------------------------------------------
# 3. CHECK REQUIRED REPORTS
# --------------------------------------------------

print("\n" + "=" * 65)
print("REPORT FILE CHECK")
print("=" * 65)

report_pass = 0

for report in REQUIRED_REPORTS:

    report_path = REPORTS / report

    if report_path.exists():

        try:
            df = pd.read_csv(report_path)

            if not df.empty:
                report_pass += 1
                print("[PASS]", report)
            else:
                errors.append(f"Empty report: {report}")
                print("[FAIL] Empty:", report)

        except Exception as e:
            errors.append(
                f"Unreadable report: {report} - {e}"
            )
            print("[FAIL] Unreadable:", report)

    else:
        errors.append(f"Missing report: {report}")
        print("[FAIL] Missing:", report)

print("\nReports available:", f"{report_pass}/{len(REQUIRED_REPORTS)}")


# --------------------------------------------------
# 4. CHECK EVALUATOR DATA FOLDER
# --------------------------------------------------

print("\n" + "=" * 65)
print("EVALUATOR DATA CHECK")
print("=" * 65)

possible_evaluator_locations = [
    BASE_DIR / "evaluator_test",
    BASE_DIR / "test",
    BASE_DIR / "evaluator",
    BASE_DIR / "data" / "evaluator_test"
]

evaluator_found = False

for folder in possible_evaluator_locations:

    if folder.exists() and folder.is_dir():
        evaluator_found = True
        print("[FOUND]", folder)
        break

if not evaluator_found:
    print("[NOT FOUND] Evaluator folder not found in common locations.")
    print("Confirm its actual location with your team.")
    print("This does not affect the training/student-test checks.")


# --------------------------------------------------
# 5. FINAL RESULT
# --------------------------------------------------

print("\n" + "=" * 65)
print("FINAL RESULT")
print("=" * 65)

all_files_pass = (
    total_trips == 90
    and gps_pass == total_trips
    and nodes_pass == total_trips
    and arcs_pass == total_trips
    and report_pass == len(REQUIRED_REPORTS)
)

if all_files_pass and not errors:

    print("[SUCCESS] ALL DATA PROCESSING CHECKS PASSED!")
    print("Training trips: 80 expected")
    print("Student testing trips: 10 expected")
    print("Processed GPS files:", gps_pass)
    print("Processed node files:", nodes_pass)
    print("Processed arc files:", arcs_pass)
    print("Required reports:", report_pass)

else:

    print("[WARNING] SOME CHECKS NEED ATTENTION.")
    print("Issues found:", len(errors))

    for error in errors:
        print("-", error)

print("\nValidation completed.")


FINAL DATA PROCESSING VALIDATION

Checking partition: train
Raw trips found: 80

Checking partition: student_test
Raw trips found: 10

PROCESSED FILE SUMMARY
Total trips: 90
GPS files:   90/90
Node files:  90/90
Arc files:   90/90

REPORT FILE CHECK
[PASS] dataset_inventory.csv
[PASS] gps_gap_analysis.csv
[PASS] gps_anomaly_details.csv
[PASS] gps_preprocessing_summary.csv
[PASS] processed_data_validation.csv
[PASS] all_road_networks_summary.csv
[PASS] road_network_validation.csv
[PASS] reference_route_validation.csv
[PASS] disconnected_route_details.csv
[PASS] route_disconnection_distances.csv

Reports available: 10/10

EVALUATOR DATA CHECK
[NOT FOUND] Evaluator folder not found in common locations.
Confirm its actual location with your team.
This does not affect the training/student-test checks.

FINAL RESULT
[SUCCESS] ALL DATA PROCESSING CHECKS PASSED!
Training trips: 80 expected
Student testing trips: 10 expected
Processed GPS files: 90
Processed node files: 90
Processed arc files: 